In [4]:
"""
    File: modello.ipynb
    Author: Andrea Macale
    Date: 2026-03-04

    Description: Notebook per la realizzazione del modello per il suggerimento ed analisi di follow-up clinici

"""

'\n    File: modello.ipynb\n    Author: Andrea Macale\n    Date: 2026-03-04\n\n    Description: Notebook per la realizzazione del modello per il suggerimento ed analisi di follow-up clinici\n\n'

# Parte 0: Importazione delle librerie

## Librerie principali

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

## Correlazione (Spearman, Pearson)

In [6]:
import scipy.stats
from scipy.stats import pearsonr, spearmanr

## VIF

In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import statsmodels.api as sm

## Modelli di ML

# Parte 1: Estrazione dei dati

## Aperta dei file del dataset

In [8]:
pos_dataset = os.path.join(Path(os.path.abspath("modello.ipynb")).parent.parent, "data", "processed")

In [9]:
file = os.path.join(pos_dataset, "patients.csv")
pazienti = pd.read_csv(file)
pazienti.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [10]:
file = os.path.join(pos_dataset, "diagnoses_icd.csv")
tipi = {
    'subject_id': str, 
    'hadm_id': str, 
    'icd_code': str, 
    'icd_version': str
}
diagnosi = pd.read_csv(file, dtype=tipi)
diagnosi.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [11]:
file = os.path.join(pos_dataset, "d_icd_diagnoses.csv")
traduzione_diagnosi = pd.read_csv(file)
traduzione_diagnosi.head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


In [12]:
file = os.path.join(pos_dataset, "mimic-cxr-2.0.0-metadata.csv")
metadati = pd.read_csv(file)
metadati.head()

,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,CHEST (PA AND LAT),PA,3056,2544,21800506,213014.531,CHEST (PA AND LAT),postero-anterior,Erect
1,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962,10000032,50414267,CHEST (PA AND LAT),LATERAL,3056,2544,21800506,213014.531,CHEST (PA AND LAT),lateral,Erect
2,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,CHEST (PA AND LAT),PA,3056,2544,21800626,165500.312,CHEST (PA AND LAT),postero-anterior,Erect
3,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c,10000032,53189527,CHEST (PA AND LAT),LATERAL,3056,2544,21800626,165500.312,CHEST (PA AND LAT),lateral,Erect
4,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,CHEST (PORTABLE AP),AP,2705,2539,21800723,80556.875,CHEST (PORTABLE AP),antero-posterior,NaN


In [13]:
lista_referti = []
ricerca = Path(os.path.join(pos_dataset, "mimic-cxr-reports", "files"))
for file_path in ricerca.rglob('*.txt'):
    study_id = file_path.stem.replace('s', '')
    subject_id = file_path.parent.name.replace('p', '')
    with open(file_path, 'r', encoding='utf-8') as file:
        testo = file.read()
    lista_referti.append({'subject_id': subject_id, 'study_id': study_id, 'testo_referto': testo})
referti = pd.DataFrame(lista_referti)
print(f"{len(referti)} referti")
referti.head()

227835 referti


,subject_id,study_id,testo_referto
0,17015391,58847767,FINAL REPORT\...
1,17015391,51918981,FINAL REPORT\...
2,17015391,54239234,FINAL REPORT\...
3,17015391,51167371,FINAL REPORT\...
4,17015391,52622416,FINAL REPORT\...


In [16]:
lista_immagini = []
ricerca = Path(os.path.join(pos_dataset, "MIMIC_SUPER_RES_24K"))
for file_path in ricerca.rglob('*.jpg'):
    dicom_id = file_path.stem
    lista_immagini.append({'dicom_id': dicom_id, 'path_immagine': str(file_path)})
immagini = pd.DataFrame(lista_immagini)
print(f"{len(immagini)} immagini")
immagini.head()

24000 immagini


,dicom_id,path_immagine
0,62456122-1887ff8c-ef1cba37-270fe1ac-c88ef7fc,/home/andy/Documenti/Tesi-Magistrale/data/proc...
1,152c3005-d0c7bf7a-4c0f7935-e82aef6e-bed96e2e,/home/andy/Documenti/Tesi-Magistrale/data/proc...
2,c4a20cd5-94e7d182-0bbca8a9-797c8bf5-7e12e050,/home/andy/Documenti/Tesi-Magistrale/data/proc...
3,064741c9-95255009-ae0c53ee-ccb54164-21b77bbb,/home/andy/Documenti/Tesi-Magistrale/data/proc...
4,2150ba84-0f87ae6f-a7132d8e-8b02fb6c-99745702,/home/andy/Documenti/Tesi-Magistrale/data/proc...
